# Prepare environment

In [1]:
%load_ext autoreload
%autoreload 2

## Init base libs

* Install libs

In [2]:
%%capture
%pip install \
    python-dotenv

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
print(os.getenv('ELASTICSEARCH_URL', 'http://elasticsearch:9200'))

http://elasticsearch:9200


## Init ElasticSearch

In [4]:
%%capture
%pip install \
    elasticsearch==8.15.1

In [5]:
import elasticsearch
from elasticsearch import Elasticsearch
from elasticsearch import helpers

# values come from ../.env through docker-compose.yml
es_url = os.getenv('ELASTICSEARCH_URL', 'http://elasticsearch:9200')
es_user = os.getenv('ELASTIC_USER')
es_password = os.getenv('ELASTIC_PASSWORD')

# credentials only when ES security is enabled (xpack.security.enabled=false in this stack)
basic_auth = (es_user, es_password) if es_user else None

es = Elasticsearch(hosts=es_url, basic_auth=basic_auth)
print("Connection to", es_url, "as", es_user or "anonymous")
es.info() # should return cluster info

Connection to http://elasticsearch:9200 as anonymous


ObjectApiResponse({'name': 'ElasticSearch', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'Y6BQGVV3Roqwt3iNUi3OLQ', 'version': {'number': '8.15.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'f97532e680b555c3a05e73a74c28afb666923018', 'build_date': '2024-10-09T22:08:00.328917561Z', 'build_snapshot': False, 'lucene_version': '9.11.1', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

# Usage

Browse every index of the cluster, for example `customers` written by the Flink job
`flink-cdc-es` (one document per row of Postgres `inventory.customers`, `_id` = customer id).

In [6]:
import pandas as pd

# Settings for the cells below
INCLUDE_SYSTEM_INDEXES = False  # True to also show indexes starting with '.'
MAX_DOCS_PER_INDEX = None       # None = all documents, or a number to cap each listing

## List all indexes

Sorted by name, with health, status, document count and size.

In [7]:
def list_indexes():
    """Names of the indexes to browse, sorted."""
    rows = es.cat.indices(format="json", h="index,health,status,docs.count,store.size", s="index")
    return [r for r in rows if INCLUDE_SYSTEM_INDEXES or not r["index"].startswith(".")]

indexes = list_indexes()
pd.DataFrame(indexes)

,index,health,status,docs.count,store.size
0,customers,yellow,open,3,37.6kb
1,customers_flinkapp,yellow,open,2,45.3kb


## Count documents of each index

Uses the count API: the exact number of searchable documents (`_cat/indices` can include
deleted or not yet refreshed documents).

In [8]:
counts = pd.DataFrame(
    [{"index": i["index"], "count": es.count(index=i["index"])["count"]} for i in indexes]
)
display(counts)
print("total:", counts["count"].sum() if not counts.empty else 0)

,index,count
0,customers,3
1,customers_flinkapp,2


total: 5


## List all documents of each index

`helpers.scan` pages through each index with the scroll API, so it isn't capped at the
10 000 hits of a single search. Each row is one document: `_id` plus its `_source` fields.

In [9]:
def list_documents(index, max_docs=MAX_DOCS_PER_INDEX):
    """All documents of `index` as a DataFrame (`_id` + `_source` fields)."""
    docs = []
    for hit in helpers.scan(es, index=index, query={"query": {"match_all": {}}}):
        docs.append({"_id": hit["_id"], **hit["_source"]})
        if max_docs is not None and len(docs) >= max_docs:
            break
    df = pd.DataFrame(docs)
    if not df.empty:
        df = df.sort_values("_id", key=lambda ids: pd.to_numeric(ids, errors="coerce").fillna(float("inf")))
        df = df.reset_index(drop=True)
    return df

documents = {}
for i in indexes:
    name = i["index"]
    documents[name] = list_documents(name)
    print(f"\n=== {name}: {len(documents[name])} document(s)")
    display(documents[name])


=== customers: 3 document(s)


,_id,id,first_name,last_name,full_name,email,street,zip,city,phone,status,registered,cdc_op,cdc_source_ts,cdc_processed_ts
0,1001,1001,Brandon,Thomas,Brandon Thomas,brandon.thomas@example.com,29 Oakwood Drive,90210,Los Angeles,(800) 555‑2100,1,2023-02-17T21:36:42Z,r,2026-11-08T15:57:12.211Z,2026-11-08T15:57:12.301Z
1,1002,1002,Sarah,Davies,Sarah Davies,sarah.davies@example.com,29 Oakwood Drive,90210,Los Angeles,NaN,1,2023-02-17T21:36:42Z,r,2026-11-08T15:57:12.211Z,2026-11-08T15:57:12.303Z
2,1005,1005,Flink,Test,Flink Test,flink.test@example.com,NaN,NaN,Paris,NaN,1,2026-11-08T16:26:40.835453Z,u,2026-11-08T16:26:53.475Z,2026-11-08T16:26:53.698Z



=== customers_flinkapp: 2 document(s)


,_id,id,first_name,last_name,full_name,email,street,zip,city,phone,status,registered,cdc_op,cdc_source_ts,cdc_processed_ts
0,1001,1001,Brandon,Thomas,Brandon Thomas,brandon.thomas@example.com,29 Oakwood Drive,90210,Los Angeles,(800) 555‑2100,1,2023-02-17T21:36:42Z,r,2026-11-08T15:57:12.211Z,2026-11-08T15:57:12.301Z
1,1002,1002,Sarah,Davies,Sarah Davies,sarah.davies@example.com,29 Oakwood Drive,90210,Los Angeles,NaN,1,2023-02-17T21:36:42Z,r,2026-11-08T15:57:12.211Z,2026-11-08T15:57:12.303Z


## Check

Every document of each index was read: the listing has as many rows as the count
(skipped when `MAX_DOCS_PER_INDEX` caps the listing).

In [10]:
for _, row in counts.iterrows():
    listed = len(documents[row["index"]])
    if MAX_DOCS_PER_INDEX is None:
        assert listed == row["count"], f"{row['index']}: count={row['count']} but listed={listed}"
    print(f"OK {row['index']}: count={row['count']} listed={listed}")

OK customers: count=3 listed=3
OK customers_flinkapp: count=2 listed=2
